<a href="https://colab.research.google.com/github/Gabriel-Roledo-ds/Analise-dos-top-10-paises-em-Inovacao-Tecnologica/blob/main/01_coleta_preparacao_modelagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Indicadores de Inovação Tecnológica
### Trabalho de Economia da Informação — Ciência de Dados - *FATEC* Ourinhos
Metodologia: CRISP-DM

## Sobre este notebook

Este notebook segue a metodologia **CRISP-DM** (Cross-Industry Standard Process for Data Mining),
um roteiro padrão de mercado para projetos de dados, dividido em 6 etapas:

1. **Entendimento do Negócio** — que perguntas queremos responder e por quê
2. **Entendimento dos Dados** — quais bases temos, o que elas contêm, como estão estruturadas
3. **Preparação dos Dados** — limpeza, padronização e consolidação das bases
4. **Modelagem** — construção dos rankings e análise de correlação entre indicadores
5. **Avaliação** — os resultados fazem sentido? respondem às perguntas da Etapa 1?
6. **Implantação** — exportação das tabelas finais e conclusões do relatório

>  Na prática, essas etapas não são 100% lineares - é comum voltar a uma etapa anterior
> ao descobrir algo novo (ex: um problema nos dados só aparece na Preparação, mesmo já
> tendo passado pela Etapa 2). Isso é normal e faz parte do processo.

# 1. Entendimento do Negócio

**Objetivo:** avaliar o nível de inovação tecnológica de diferentes países, comparando
esforço (investimento) com resultado (produção de conhecimento/tecnologia).

**Perguntas que este notebook busca responder:**
- Quais países mais investem em P&D e possuem mais pesquisadores?
- Quais países mais geram patentes, marcas, desenhos industriais e exportações de alta tecnologia?
- Existe relação entre esforço e resultado?
- Como o Brasil se posiciona frente aos líderes?
- Como isso evoluiu nos últimos 5 recortes de 5 em 5 anos?

**Critério de sucesso:** base consolidada e confiável, com rankings Top 10 por indicador/ano,
sustentando as tabelas da Fase 1 (22/09/2026) e as conclusões da Fase 2 (09/11/2026).

##Setup e Imports

In [169]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)

CAMINHO = "/content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/"

anos_bm = [2021, 2016, 2011, 2006, 2001]
anos_wipo = [2024, 2019, 2014, 2009, 2004]

#Estabeleço a conexão com o drive
#Faço o import da biblioteca para manipular a base
#Defino variáveis com os anos da frequência 5 (Tabela com frequências no guia do trabalho)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Funções

In [170]:


def carregar_indicador_bm(caminho_dados, caminho_metadata_country, nome_indicador, anos):
    """Carrega um indicador do Banco Mundial em formato long, sem agregados regionais."""
    df = pd.read_csv(caminho_dados, skiprows=4)

    meta = pd.read_csv(caminho_metadata_country)
    paises_validos = meta.loc[meta["Region"].notna(), "Country Code"]
    df = df[df["Country Code"].isin(paises_validos)]

    df_long = df.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=[str(a) for a in anos],
        var_name="Ano",
        value_name="Valor"
    )
    df_long["Ano"] = df_long["Ano"].astype(int)
    df_long["Indicador"] = nome_indicador
    return df_long

#____________________________________________________________________


def diagnostico(df):
    """Tabela resumo: tipo, % de nulos e valores únicos por coluna."""
    return pd.DataFrame({
        "Tipo": df.dtypes,
        "Nulos (%)": (df.isna().mean() * 100).round(2),
        "Valores únicos": df.nunique()
    })

#____________________________________________________________________

def diagnostico_completo(df, coluna_valor="Valor", coluna_ano="Ano"):
    """Checklist de confiabilidade — rodar só após o dado estar em formato long."""
    print("== Tipos e nulos ==")
    display(diagnostico(df))

    print("\n== Estatísticas descritivas ==")
    display(df[coluna_valor].describe())

    print("\n== Cobertura por ano (nº de países com dado) ==")
    display(df.groupby(coluna_ano)[coluna_valor].count())

    print("\n== Duplicatas país+ano ==")
    print(df.duplicated(subset=["Country Code", coluna_ano]).sum())

#______________________________________________________________________


def carregar_indicador_wipo(caminho_dados, nome_indicador_wipo, nome_indicador, anos):
    """
    Carrega um indicador da base WIPO (patentes/marcas/desenhos) em formato long.
    nome_indicador_wipo: texto exato da coluna 'Statistics' que identifica o indicador desejado
    nome_indicador: rótulo que usaremos no dataframe consolidado
    """
    df = pd.read_csv(caminho_dados, skiprows=6, index_col=False)

    df = df[df["Statistics"] == nome_indicador_wipo]

    df_long = df.melt(
        id_vars=["Origin", "Origin (Code)"],
        value_vars=[str(a) for a in anos],
        var_name="Ano",
        value_name="Valor"
    )
    df_long["Ano"] = df_long["Ano"].astype(int)
    df_long["Indicador"] = nome_indicador
    df_long = df_long.rename(columns={"Origin": "Country Name", "Origin (Code)": "Country Code"})

    return df_long

#_______________________________________________________________________________

def top10_por_indicador_ano(df, indicador, ano):
    """Retorna o Top 10 países de um indicador, em um ano específico, ordenado do maior para o menor valor."""
    filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
    return (
        df[filtro]
        .dropna(subset=["Valor"])
        .sort_values("Valor", ascending=False)
        .head(10)
        [["Country Name PT", "Valor"]]
        .rename(columns={"Country Name PT": "País"})
        .reset_index(drop=True)
    )

#_______________________________________________________________________________

!pip install pycountry -q

import pycountry
from babel import Locale

def traduzir_pais(codigo_iso):
    """Traduz um código de país (ISO Alpha-2 ou Alpha-3) para o nome em português."""
    if not isinstance(codigo_iso, str):
        return None  # código ausente (ex: Namíbia com "NA" lido como nulo)

    if len(codigo_iso) == 3:
        pais = pycountry.countries.get(alpha_3=codigo_iso)
        codigo_alpha2 = pais.alpha_2 if pais else None
    else:
        codigo_alpha2 = codigo_iso  # já está em Alpha-2

    if codigo_alpha2 is None:
        return codigo_iso

    loc = Locale('pt')
    return loc.territories.get(codigo_alpha2, codigo_iso)

#_______________________________________________________________________________

def paises_estaveis_no_top10(top10_evolucao, indicador):
    """
    Retorna os países que aparecem no Top 10 do indicador em TODOS os anos disponíveis,
    e também a lista de anos considerada.
    """
    anos = list(top10_evolucao[indicador].keys())
    conjuntos_por_ano = [set(top10_evolucao[indicador][ano]["País"]) for ano in anos]
    paises_sempre_presentes = set.intersection(*conjuntos_por_ano)
    return sorted(paises_sempre_presentes), anos


def resumo_evolucao_top10(top10_evolucao, indicador):
    """Imprime um resumo: países estáveis, e quais entraram/saíram entre o primeiro e o último ano."""
    anos = sorted(top10_evolucao[indicador].keys())
    primeiro_ano, ultimo_ano = anos[0], anos[-1]

    paises_primeiro = set(top10_evolucao[indicador][primeiro_ano]["País"])
    paises_ultimo = set(top10_evolucao[indicador][ultimo_ano]["País"])

    estaveis, _ = paises_estaveis_no_top10(top10_evolucao, indicador)
    saíram = paises_primeiro - paises_ultimo
    entraram = paises_ultimo - paises_primeiro

    print(f"=== {indicador} ===")
    print(f"Países no Top 10 em TODOS os {len(anos)} anos ({len(estaveis)}): {estaveis}")
    print(f"Saíram do Top 10 (estavam em {primeiro_ano}, não estão em {ultimo_ano}): {sorted(saíram)}")
    print(f"Entraram no Top 10 (não estavam em {primeiro_ano}, estão em {ultimo_ano}): {sorted(entraram)}")

#_______________________________________________________________________________

def matriz_posicao_top10(top10_evolucao, indicador):
    """
    Constrói uma matriz País x Ano, mostrando a posição (1-10) de cada país
    no Top 10 daquele ano. Células vazias (—) indicam que o país não esteve
    no Top 10 naquele ano.
    """
    anos = sorted(top10_evolucao[indicador].keys())

    # Junta todos os países que apareceram em pelo menos 1 ano (união dos 5 conjuntos)
    todos_paises = set()
    for ano in anos:
        todos_paises.update(top10_evolucao[indicador][ano]["País"])

    # Monta a matriz, linha por linha (um país de cada vez)
    linhas = []
    for pais in sorted(todos_paises):
        linha = {"País": pais}
        for ano in anos:
            tabela_ano = top10_evolucao[indicador][ano]
            posicao = tabela_ano[tabela_ano["País"] == pais].index

            if len(posicao) > 0:
                linha[ano] = posicao[0] + 1  # +1 porque o index começa em 0
            else:
                linha[ano] = "—"
        linhas.append(linha)

    matriz = pd.DataFrame(linhas).set_index("País")
    return matriz

#_______________________________________________________________________________

INDICADORES_ABSOLUTOS = [
    "Pedidos de patentes bm",
    "Pedidos de patentes wipo",
    "Pedidos de marcas wipo",
    "Pedidos de desenhos industriais wipo",
]

INDICADORES_PERCENTUAIS = [
    "Gasto em P&D bm (% do PIB)",
    "Exportações de alta tecnologia bm (%)",
]

def top10_com_total_mundial(df, indicador, ano):
    """
    Gera o Top 10 de um indicador/ano, com coluna 'Ranking' (1-10).
    Para indicadores absolutos, adiciona '% da produção mundial' e uma linha
    final 'Total mundial'.
    """
    top10 = top10_por_indicador_ano(df, indicador, ano)
    top10.insert(0, "Ranking", range(1, len(top10) + 1))

    if indicador not in INDICADORES_ABSOLUTOS:
        return top10

    filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
    total_mundial = df[filtro]["Valor"].sum()

    top10 = top10.copy()
    top10["% da produção mundial"] = (top10["Valor"] / total_mundial * 100).round(2)

    linha_total = pd.DataFrame([{
        "Ranking": "—",
        "País": "Total mundial",
        "Valor": total_mundial,
        "% da produção mundial": 100.00
    }])

    return pd.concat([top10, linha_total], ignore_index=True)


def formatar_para_exibicao(tabela, indicador):
    """
    Formata os números de uma tabela já pronta para exibição final:
    - Indicadores absolutos: 'Valor' vira texto com separador de milhar (1.796.738)
    - Indicadores percentuais: 'Valor' vira texto com % (5,76%)
    - '% da produção mundial' sempre vira texto com % (48,53%)
    """
    tabela = tabela.copy()

    if indicador in INDICADORES_ABSOLUTOS:
        tabela["Valor"] = tabela["Valor"].apply(lambda v: f"{v:,.0f}".replace(",", "."))
        if "% da produção mundial" in tabela.columns:
            tabela["% da produção mundial"] = tabela["% da produção mundial"].apply(
                lambda v: f"{v:.2f}".replace(".", ",") + "%"
            )
    elif indicador in INDICADORES_PERCENTUAIS:
        tabela["Valor"] = tabela["Valor"].apply(lambda v: f"{v:.2f}".replace(".", ",") + "%")
    else:
        # Pesquisadores em P&D: número absoluto, mas "por milhão hab.", não % nem contagem bruta de país
        tabela["Valor"] = tabela["Valor"].apply(lambda v: f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

    return tabela

#________________________________________________________________________________

!pip install pycountry_convert -q
import pycountry_convert as pc

NOMES_CONTINENTES = {
    "AF": "África",
    "AS": "Ásia",
    "EU": "Europa",
    "NA": "América do Norte",
    "SA": "América do Sul",
    "OC": "Oceania",
}

def continente_do_pais(codigo_iso3):
    """Retorna o continente geográfico de um país, a partir do código ISO Alpha-3."""
    if not isinstance(codigo_iso3, str):
        return None  # código ausente (ex: Namíbia)

    pais = pycountry.countries.get(alpha_3=codigo_iso3) if len(codigo_iso3) == 3 else None
    codigo_alpha2 = pais.alpha_2 if pais else (codigo_iso3 if len(codigo_iso3) == 2 else None)

    if codigo_alpha2 is None:
        return None

    try:
        codigo_continente = pc.country_alpha2_to_continent_code(codigo_alpha2)
        return NOMES_CONTINENTES.get(codigo_continente, None)
    except KeyError:
        return None  # código não reconhecido pela biblioteca (raro, mas acontece)

#______________________________________________________________________________

def carregar_nivel_renda(caminho_metadata_country):
    """
    Carrega o nível de renda (IncomeGroup) de cada país a partir de um arquivo
    de metadata do Banco Mundial. Traduz as categorias para português.
    """
    meta = pd.read_csv(caminho_metadata_country)
    meta = meta[["Country Code", "IncomeGroup"]].dropna(subset=["IncomeGroup"])

    traducao_renda = {
        "High income": "Renda alta (desenvolvido)",
        "Upper middle income": "Renda média-alta (emergente)",
        "Lower middle income": "Renda média-baixa (emergente)",
        "Low income": "Renda baixa (subdesenvolvido)",
    }
    meta["Nível de Renda"] = meta["IncomeGroup"].map(traducao_renda)

    return meta.rename(columns={"Country Code": "Código País"})[["Código País", "Nível de Renda"]]

#_______________________________________________________________________________

def ranking_continentes(df, indicador, ano):
    """
    Agrega o valor de um indicador por continente, num ano específico,
    ordenado do maior para o menor. Para indicadores absolutos, soma os valores;
    para percentuais/per-capita, calcula a MÉDIA (soma não faz sentido nesses casos).
    """
    filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
    dados = df[filtro].dropna(subset=["Valor", "Continente"])

    if dados.empty:
        print(f"⚠️ Sem dado disponível para '{indicador}' em {ano}.")
        return pd.DataFrame(columns=["Ranking", "Continente", "Valor"])

    if indicador in INDICADORES_ABSOLUTOS:
        agregado = dados.groupby("Continente")["Valor"].sum().reset_index()
        agregado = agregado.sort_values("Valor", ascending=False).reset_index(drop=True)

        total_mundial = agregado["Valor"].sum()
        agregado["% da produção mundial"] = (agregado["Valor"] / total_mundial * 100).round(2)
    else:
        agregado = dados.groupby("Continente")["Valor"].mean().reset_index()
        agregado = agregado.sort_values("Valor", ascending=False).reset_index(drop=True)

    agregado.insert(0, "Ranking", range(1, len(agregado) + 1))
    return agregado

#______________________________________________________________________________

def codigo_para_alpha3(codigo):
    """Converte um código de país (Alpha-2 ou Alpha-3) para Alpha-3 padronizado."""
    if not isinstance(codigo, str):
        return None
    if len(codigo) == 3:
        return codigo  # já é Alpha-3
    pais = pycountry.countries.get(alpha_2=codigo)
    return pais.alpha_3 if pais else None

#______________________________________________________________________________

def top10_continente(df, indicador, ano, continente):
    """
    Retorna o Top 10 países de um indicador/ano, dentro de um continente específico,
    com a % de participação de cada país sobre o total daquele continente.
    """
    filtro = (
        (df["Indicador"] == indicador) &
        (df["Ano"] == ano) &
        (df["Continente"] == continente)
    )
    dados = df[filtro].dropna(subset=["Valor"]).sort_values("Valor", ascending=False).reset_index(drop=True)

    if dados.empty:
        return pd.DataFrame(columns=["Ranking", "País", "Valor", "% do continente"])

    top10 = dados.head(10)[["País", "Valor"]].copy()

    if indicador in INDICADORES_ABSOLUTOS:
        total_continente = dados["Valor"].sum()
        top10["% do continente"] = (top10["Valor"] / total_continente * 100).round(2)

    top10.insert(0, "Ranking", range(1, len(top10) + 1))
    return top10

#________________________________________________________________________________

def ranking_nivel_renda(df, indicador, ano):
    """
    Agrega o valor de um indicador por nível de renda, num ano específico,
    ordenado do maior para o menor. Mesma lógica de ranking_continentes:
    soma para indicadores absolutos, média para percentuais/per-capita.
    """
    filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
    dados = df[filtro].dropna(subset=["Valor", "Nível de Renda"])

    if dados.empty:
        print(f"⚠️ Sem dado disponível para '{indicador}' em {ano}.")
        return pd.DataFrame(columns=["Ranking", "Nível de Renda", "Valor"])

    if indicador in INDICADORES_ABSOLUTOS:
        agregado = dados.groupby("Nível de Renda")["Valor"].sum().reset_index()
        agregado = agregado.sort_values("Valor", ascending=False).reset_index(drop=True)

        total_mundial = agregado["Valor"].sum()
        agregado["% da produção mundial"] = (agregado["Valor"] / total_mundial * 100).round(2)
    else:
        agregado = dados.groupby("Nível de Renda")["Valor"].mean().reset_index()
        agregado = agregado.sort_values("Valor", ascending=False).reset_index(drop=True)

    agregado.insert(0, "Ranking", range(1, len(agregado) + 1))
    return agregado


# 2. Entendimento dos Dados

Fontes utilizadas:
- **[Gasto em P&D (% do PIB)](https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS )** — Banco Mundial
- **[Pesquisadores em P&D](https://data.worldbank.org/indicator/SP.POP.SCIE.RD.P6 )** — Banco Mundial
- **[Pedidos de patentes](https://data.worldbank.org/indicator/IP.PAT.RESD)** — Banco Mundial
- **[Exportações de alta tecnologia](https://data.worldbank.org/indicator/TX.VAL.TECH.MF.ZS)** — Banco Mundial
- **[Patentes, Marcas e Desenhos Industriais](https://www3.wipo.int/ipstats/ips-search/countryprofiles)** — WIPO

Os 4 indicadores do Banco Mundial vêm em formato *wide* (anos em colunas), com metadados
de país e indicador em arquivos separados. A base da WIPO tem formato ainda a ser explorado.

Para facilitar o entendimento, os arquivos foram renomeados com titulos descritivos para esse notebook.

##2.1 Exploração inicial: Gasto em P&D (% do PIB), cru

In [171]:

df_teste = pd.read_csv(
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    skiprows=4  # pula os metadados soltos antes da tabela real
)


df_teste.shape

(265, 71)

In [172]:
df_teste.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



*   Formato wide — cada ano é uma coluna, cada país é uma linha. Será feito um `.melt` para transformar as colunas de ano em duas colunas: `Ano` e `Valor`. O formato long (Country Name / Country Code / Ano / Valor) vai facilitar as futuras comparações entre tabelas.

*   Existem "países" que na verdade são agregados regionais/blocos de renda (World, OECD members...), que precisam ser removidos antes do ranking.

*   Foi necessário o uso de `skiprows` porque o pandas lia o título no lugar do cabeçalho da tabela.

##2.2 Exploração inicial: WIPO (Patentes, Marcas, Desenhos Industriais), cru

In [173]:
df_wipo_bruto = pd.read_csv(
    CAMINHO + "Indicadores_PeD/patents_trademarks_industrial_design.csv",
    skiprows=6,
    index_col=False
)

df_wipo_bruto.shape



(522, 25)

In [174]:
df_wipo_bruto.columns.tolist()

['Origin',
 'Origin (Code)',
 'Office',
 'Statistics',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024']

In [175]:
df_wipo_bruto.head(5)

,Origin,Origin (Code),Office,Statistics,2004,2005,2006,2007,2008,2009,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Albania,AL,Total,1.1 - Patent - Total patent applications,NaN,1.00,NaN,NaN,NaN,4.00,...,16.00,37.00,18.00,21.00,12.00,NaN,32.00,28.00,28.00,NaN
1,Albania,AL,Total,2.1 - Trademark - Total classes in trademark a...,NaN,NaN,NaN,NaN,NaN,NaN,...,966.00,1019.00,1310.00,1775.00,1527.00,1630.00,2132.00,1523.00,2155.00,2440.00
2,Albania,AL,Total,3.1 - Industrial design - Total designs in app...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,58.00,58.00,50.00,234.00,31.00
3,Algeria,DZ,Total,1.1 - Patent - Total patent applications,59.00,65.00,59.00,88.00,NaN,NaN,...,107.00,112.00,158.00,162.00,119.00,170.00,280.00,483.00,1412.00,1117.00
4,Algeria,DZ,Total,2.1 - Trademark - Total classes in trademark a...,NaN,NaN,NaN,NaN,NaN,NaN,...,14638.00,NaN,8580.00,7350.00,9816.00,12561.00,11445.00,11868.00,14314.00,16033.00


##2.3 Identificar coluna de indicador na base WIPO

In [176]:
for col in df_wipo_bruto.columns:
    n_unicos = df_wipo_bruto[col].nunique()
    if n_unicos <= 15:
        print(f"{col}: {n_unicos} valores únicos -> {df_wipo_bruto[col].unique()}")

Office: 1 valores únicos -> ['Total']
Statistics: 3 valores únicos -> ['1.1 - Patent - Total patent applications'
 '2.1 - Trademark - Total classes in trademark applications'
 '3.1 - Industrial design - Total designs in applications']


In [177]:
print("Total de origens únicas:", df_wipo_bruto["Origin"].nunique())
print(sorted(df_wipo_bruto["Origin"].unique()))

Total de origens únicas: 187
['Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'China, Hong Kong SAR', 'China, Macao SAR', 'Colombia', 'Comoros', 'Congo', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Republic', "Côte d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Estonia', 'Eswatini', 'Ethiopia', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'G

## 2.4 Observações — estrutura da base WIPO

### Estrutura da base WIPO
- Coluna que identifica o indicador: `Statistics`
- Valores encontrados: `1.1 - Patent - Total patent applications`, `2.1 - Trademark - Total classes in trademark applications`, `3.1 - Industrial design - Total designs in applications`
- Formato da base: wide (anos 2004-2024 em colunas), com os 3 indicadores empilhados como linhas (3 linhas por país) — diferente do Banco Mundial, que tem 1 arquivo por indicador
- Coluna de país: `Origin` (nome) / `Origin (Code)` (código ISO2, ex: "AL") — coluna `Office` é sempre "Total", não carrega informação útil aqui

##2.5 Divergência de nomes de país entre fontes


A base WIPO (187 origens, sem agregados regionais) usa nomenclatura diferente do
Banco Mundial para vários países. Exemplos:

| Banco Mundial | WIPO |
|---|---|
| United States | United States of America |
| Korea, Rep. | Republic of Korea |
| Iran, Islamic Rep. | Iran (Islamic Republic of) |
| Egypt, Arab Rep. | Egypt |
| Venezuela, RB | Venezuela (Bolivarian Republic of) |
| Czechia | Czech Republic |
| Bahamas, The | Bahamas |

**Implicação para a consolidação:** um `merge` direto por nome de país vai falhar
silenciosamente nesses casos (linhas não vão casar). Duas opções a avaliar na Etapa 3:
- Usar o **código de país (ISO)** como chave de junção, quando disponível e no mesmo padrão nas duas fontes
- Construir um dicionário manual de correspondência de nomes divergentes

#3. Preparação dos Dados

Nesta etapa:
- Removemos agregados regionais/blocos de renda (não são países)
- Convertemos os 4 indicadores do Banco Mundial de *wide* para *long*
- Filtramos apenas os 5 anos de referência
- Adequamos a base WIPO ao mesmo formato
- Rodamos diagnóstico de confiabilidade em cada indicador antes de consolidar

##3.1 Carregamento e limpeza: Gasto em P&D (% do PIB)

In [178]:

df_pd_pib = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/metadata_country_pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    "Gasto em P&D bm (% do PIB)",   # <- alterado
    anos_bm
)

df_pd_pib.shape
df_pd_pib["Country Name"].nunique()

217

In [179]:
df_pd_pib.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Gasto em P&D bm (% do PIB)
1,Afghanistan,AFG,2021,NaN,Gasto em P&D bm (% do PIB)
2,Angola,AGO,2021,NaN,Gasto em P&D bm (% do PIB)
3,Albania,ALB,2021,0.19,Gasto em P&D bm (% do PIB)
4,Andorra,AND,2021,NaN,Gasto em P&D bm (% do PIB)


##3.2 Carregamento e limpeza: Pesquisadores em P&D

In [180]:
df_pesquisadores = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/Pesquisadores envolvidos em P&D/pesquisadores_envolvidos_em_P&D.csv",
    CAMINHO + "Indicadores_PeD/Pesquisadores envolvidos em P&D/metadata_country_pesquisadores_envolvidos_em_P&D.csv",
    "Pesquisadores em P&D bm (por milhão hab.)",   # <- alterado
    anos_bm
)
df_pesquisadores.shape
df_pesquisadores["Country Name"].nunique()

217

In [181]:
df_pesquisadores.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Pesquisadores em P&D bm (por milhão hab.)
1,Afghanistan,AFG,2021,NaN,Pesquisadores em P&D bm (por milhão hab.)
2,Angola,AGO,2021,NaN,Pesquisadores em P&D bm (por milhão hab.)
3,Albania,ALB,2021,449.10,Pesquisadores em P&D bm (por milhão hab.)
4,Andorra,AND,2021,NaN,Pesquisadores em P&D bm (por milhão hab.)


##3.3 Carregamento e limpeza: Pedidos de patentes

In [182]:
df_patentes_resd = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/pedidos_de_patente/pedidos_de_patente.csv",
    CAMINHO + "Indicadores_PeD/pedidos_de_patente/metadata_country_pedidos_de_patente.csv",
    "Pedidos de patentes bm",   # <- alterado
    anos_bm
)

df_patentes_resd.shape
df_patentes_resd["Country Name"].nunique()

217

In [183]:
df_patentes_resd.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Pedidos de patentes bm
1,Afghanistan,AFG,2021,NaN,Pedidos de patentes bm
2,Angola,AGO,2021,NaN,Pedidos de patentes bm
3,Albania,ALB,2021,23.00,Pedidos de patentes bm
4,Andorra,AND,2021,3.00,Pedidos de patentes bm


##3.4 Carregamento e limpeza: Exportações de alta tecnologia

In [184]:

df_exportacoes = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/exportações de alta tecnologia/exportações_de_alta_tecnologia.csv",
    CAMINHO + "Indicadores_PeD/exportações de alta tecnologia/metadata_exportações_de_alta tecnologia.csv",
    "Exportações de alta tecnologia bm (%)",   # <- alterado
    anos_bm
)

df_exportacoes.shape
df_exportacoes["Country Name"].nunique()

217

In [185]:
df_exportacoes.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,4.82,Exportações de alta tecnologia bm (%)
1,Afghanistan,AFG,2021,NaN,Exportações de alta tecnologia bm (%)
2,Angola,AGO,2021,16.13,Exportações de alta tecnologia bm (%)
3,Albania,ALB,2021,0.42,Exportações de alta tecnologia bm (%)
4,Andorra,AND,2021,24.38,Exportações de alta tecnologia bm (%)


##3.5 Diagnóstico: Gasto em P&D (% do PIB)

In [186]:
diagnostico_completo(df_pd_pib)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,59.63,438
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,438.00
mean,0.99
std,1.01
min,0.01
25%,0.23
50%,0.62
75%,1.43
max,5.76



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,82
2006,85
2011,92
2016,91
2021,88



== Duplicatas país+ano ==
0


**Estatística:**
___

**count 438**

Quantidade de valores não nulos usados no cálculo. Você tem 5 anos × 217 países = 1085 combinações possíveis, mas só 438 têm dado de verdade (o resto é NaN) — bate com os 59,63% de nulos que vimos no diagnóstico de tipos.
___

**mean 0.99**

A média — soma de todos os 438 valores, dividida por 438. Em média, os países da amostra gastam 0,99% do PIB em P&D.
___
**std 1.01 (desvio padrão)**

Mede o quanto os valores variam em torno da média. Um desvio padrão de 1,01, quase do tamanho da própria média (0,99), indica que os dados são bem dispersos — tem muita diferença entre os países, não é um grupo homogêneo. Isso já é esperado: países como Israel/Coreia do Sul gastam bem mais que 3% do PIB, enquanto muitos países em desenvolvimento gastam frações de 1%.
___
**min 0.0126**

O menor valor de gasto em P&D encontrado na base inteira (entre todos os 438 registros) — 0,0126% do PIB, praticamente nada.
___
**max 5.76**

O maior valor encontrado — um país (não sabemos qual ainda, sem olhar) chegou a investir quase 6% do PIB em P&D num dos 5 anos-base. Isso é um valor alto mas plausível (Israel e Coreia do Sul historicamente ficam nessa faixa), então não é um outlier suspeito de erro.
___

**25%, 50%, 75% (quartis)**


   Esses três dividem os 438 valores em 4 partes iguais, depois de ordenados do menor para o maior

* 25% = 0.233 → 25% dos registros têm gasto abaixo de 0,233% do PIB (ou seja, um quarto dos países-ano são bem pouco investidores)

* 50% = 0.617 (a mediana) → metade dos registros está abaixo desse valor, metade acima. Repare que a mediana (0,617) é bem menor que a média (0,99) — isso é um sinal importante: significa que a distribuição é assimétrica, puxada para cima por poucos países que investem muito (tipo Israel, Coreia do Sul), enquanto a maioria dos países fica bem abaixo da média.

* 75% = 1.43 → 75% dos registros estão abaixo de 1,43%; só o quarto "de cima" (os 25% mais investidores) ultrapassa esse valor.

___

**Padrão ao longo do tempo:** a cobertura sobe de 2001 (82) até 2011 (92), e depois cai levemente até 2021 (88). Isso é um dado interessante para o relatório: não indica necessariamente que menos países investem em P&D em 2021 — indica que menos países reportaram esse dado em 2021

##3.6 Diagnóstico: Pesquisadores em P&D

In [187]:
diagnostico_completo(df_pesquisadores)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,67.10,357
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,357.00
mean,2209.66
std,2105.93
min,5.94
25%,454.80
50%,1594.22
75%,3558.20
max,9071.45



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,56
2006,64
2011,76
2016,78
2021,83



== Duplicatas país+ano ==
0


### Observações — Pesquisadores em P&D (por milhão hab.)

- **Cobertura:** 357 de 1085 combinações possíveis têm dado (67,1% de nulos) — cobertura ainda pior que Gasto em P&D (59,63%)
- **Média:** 2.209,7 | **Mediana:** 1.594,2 — mediana menor que a média, distribuição assimétrica (poucos países com número muito alto de pesquisadores puxam a média para cima)
- **Dispersão (desvio padrão):** 2.105,9 — quase do tamanho da própria média, dados muito dispersos entre países
- **Mínimo / Máximo:** 5,9 / 9.071,5 pesquisadores por milhão de hab. — intervalo enorme, mas plausível (países com pouca estrutura de pesquisa vs. países como Coreia do Sul/Israel com forte base científica)
- **Cobertura por ano:** cresce de forma constante — 56 (2001) → 64 (2006) → 76 (2011) → 78 (2016) → 83 (2021). Diferente do Gasto em P&D (que caiu no fim), aqui a tendência é de melhora contínua na disponibilidade do dado ao longo do tempo
- **Duplicatas:** 0 — melt correto

**Ponto de atenção:** a cobertura em 2001 (56 países) é bem menor que em 2021 (83) — quase 50% a mais de países reportando no fim do período. Isso é ainda mais relevante que no indicador de P&D, porque compromete comparações do Top 10 entre os anos mais antigos e recentes: o "pool" de concorrência em 2001 é bem menor.

##3.7 Diagnóstico: Pedidos de patentes

In [188]:
diagnostico_completo(df_patentes_resd)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,51.61,374
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,525.00
mean,14369.82
std,92219.91
min,1.00
25%,32.00
50%,212.00
75%,1300.00
max,1426644.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,88
2006,96
2011,106
2016,122
2021,113



== Duplicatas país+ano ==
0


### Observações — Pedidos de patentes

- **Cobertura:** 525 de 1085 combinações possíveis têm dado (48,39% de nulos) — a melhor cobertura entre os 3 indicadores vistos até agora

- **Média:** 14.369,8 | **Mediana:** 212 — diferença enorme entre média e mediana, sinal de distribuição extremamente assimétrica: uns poucos países (provavelmente China, EUA, Japão) registram centenas de milhares de pedidos, enquanto a maioria fica na casa das centenas

- **Dispersão (desvio padrão):** 92.219,9 — muito maior que a própria média, confirma a forte concentração em poucos países

- **Mínimo / Máximo:** 1 / 1.426.644 — intervalo gigantesco (de 1 único pedido a mais de 1,4 milhão), plausível dado que esse indicador é contagem absoluta (não % nem por-milhão-de-habitante como os anteriores), então países grandes dominam naturalmente

- **Cobertura por ano:** cresce de 88 (2001) para 122 (2016), cai um pouco para 113 (2021) — mesma ressalva de cobertura desigual entre anos-base
- **Duplicatas:** 0 — melt correto



**Ponto de atenção para consolidação:** este indicador de patentes é **absoluto** (contagem bruta), enquanto Gasto em P&D e Pesquisadores são **relativos** (% ou por milhão de habitantes). Isso significa que comparar diretamente "quem tem mais patentes" favorece países grandes (população/economia), diferente dos outros dois indicadores, que já são normalizados. Vale mencionar essa diferença de natureza entre indicadores na Fase 2, ao interpretar correlações.

##3.8 Diagnóstico: Exportações de alta tecnologia

In [189]:
diagnostico_completo(df_exportacoes)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,56.22,470
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,475.00
mean,10.31
std,12.22
min,0.00
25%,1.72
50%,6.43
75%,15.33
max,70.55



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,0
2006,0
2011,143
2016,166
2021,166



== Duplicatas país+ano ==
0


###  Observações — Exportações de alta tecnologia (%)

- **Cobertura:** 475 de 1085 combinações possíveis têm dado (56,22% de nulos)

- **Média:** 10,31% | **Mediana:** 6,43% — mediana bem menor que a média, distribuição assimétrica (poucos países exportadores de alta tecnologia muito fortes puxam a média para cima)

- **Dispersão (desvio padrão):** 12,22 — maior que a própria média, dados bastante dispersos

- **Mínimo / Máximo:** 0% / 70,55% — intervalo plausível (0% para países sem exportação de tecnologia, até um país fortemente especializado em tech)

- **Duplicatas:** 0 — melt correto

**Problema real de cobertura:** diferente dos outros indicadores (que tinham cobertura desigual mas presente em todos os anos), este indicador tem **ZERO países com dado em 2001 e 2006**. Cobertura só começa a existir a partir de 2011 (143 países), 2016 (166) e 2021 (166).

**O que isso significa na prática:** não dá para montar um Top 10 de Exportações de alta tecnologia para os anos 2001 e 2006 — simplesmente não existe dado nesse indicador para essas datas na base do Banco Mundial. Isso não é erro do nosso código (o `carregar_indicador_bm` e o filtro de anos estão corretos) — é uma limitação real da fonte de dados: esse indicador provavelmente começou a ser sistematicamente coletado/publicado só a partir dos anos 2010.

**Ação necessária:** avaliar se, para este indicador específico, o grupo apresenta o Top 10 apenas para 2011/2016/2021 (com uma nota explicando a ausência de dados anteriores), ou se buscamos uma fonte alternativa para os anos faltantes. Vale confirmar isso com o professor/guia do trabalho, já que os outros 4 indicadores têm os 5 anos completos.

##3.9 Carregamento e limpeza: Patentes, Marcas e Desenhos Industriais (WIPO)

In [190]:
CAMINHO_WIPO = CAMINHO + "Indicadores_PeD/patents_trademarks_industrial_design.csv"

df_patentes_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "1.1 - Patent - Total patent applications",
    "Pedidos de patentes wipo",   # <- alterado
    anos_wipo
)

df_marcas_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "2.1 - Trademark - Total classes in trademark applications",
    "Pedidos de marcas wipo",   # <- alterado
    anos_wipo
)

df_desenhos_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "3.1 - Industrial design - Total designs in applications",
    "Pedidos de desenhos industriais wipo",   # <- alterado
    anos_wipo
)

for nome, d in [("Patentes WIPO", df_patentes_wipo), ("Marcas", df_marcas_wipo),
                ("Desenhos industriais", df_desenhos_wipo)]:
    print(nome, "->", d.shape, "| países únicos:", d["Country Name"].nunique())

Patentes WIPO -> (880, 5) | países únicos: 176
Marcas -> (915, 5) | países únicos: 183
Desenhos industriais -> (815, 5) | países únicos: 163


##3.10 Diagnóstico: Pedidos de patentes (WIPO)

In [191]:
diagnostico_completo(df_patentes_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,176
Country Code,object,0.57,175
Ano,int64,0.00,5
Valor,float64,27.84,430
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,635.00
mean,20040.75
std,112429.49
min,1.00
25%,27.00
50%,250.00
75%,2037.00
max,1796738.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,113
2009,100
2014,139
2019,137
2024,146



== Duplicatas país+ano ==
0


###  Observações — Pedidos de patentes (WIPO)

- **Cobertura:** 635 de 880 combinações possíveis têm dado (27,84% de nulos) — melhor cobertura entre todos os indicadores vistos até agora (mesmo indicador do Banco Mundial tinha 48,39% de nulos)

- **Média:** 20.040,8 | **Mediana:** 250 — diferença extrema entre média e mediana, distribuição fortemente assimétrica (poucos países concentram a maior parte dos pedidos — provavelmente China/EUA)

- **Dispersão (desvio padrão):** 112.429,5 — mais de 5x a média, confirma concentração extrema em poucos países

- **Mínimo / Máximo:** 1 / 1.796.738 — intervalo ainda maior que o mesmo indicador no Banco Mundial (1.426.644), plausível pois são fontes/metodologias diferentes de contagem

- **Cobertura por ano:** oscila entre 100-146 países, sem tendência clara de queda ou crescimento constante — 113 (2004) → 100 (2009) → 139 (2014) → 137 (2019) → 146 (2024)

- **Duplicatas:** 0 — melt correto

**Nulos em `Country Code`:** diferente do Banco Mundial (nunca tinha nulo em Country Code), aqui 0,57% das linhas têm código de país ausente. Precisa investigar: pode ser um "Country Name" sem correspondência de código no arquivo original da WIPO (ex: território ou nome incomum). Vale rodar uma checagem antes de seguir:



In [192]:
df_patentes_wipo[df_patentes_wipo["Country Code"].isna()]["Country Name"].unique()

array(['Namibia'], dtype=object)

###  Nota — Namíbia com Country Code nulo (não corrigido)

Identificado que o código "NA" da Namíbia é lido como nulo pelo pandas. Não afeta o
escopo deste trabalho, já que o objetivo é somente o Top 10 de cada indicador — caso apareça no top 10, será preciso tratar.


##3.11 Diagnóstico: Pedidos de marcas



In [193]:
diagnostico_completo(df_marcas_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,183
Country Code,object,0.55,182
Ano,int64,0.00,5
Valor,float64,33.77,583
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,606.00
mean,71869.05
std,456370.77
min,1.00
25%,929.25
50%,6436.50
75%,29589.25
max,7904365.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,90
2009,79
2014,134
2019,148
2024,155



== Duplicatas país+ano ==
0


###  Observações — Pedidos de marcas

- **Cobertura:** 606 de 915 combinações possíveis têm dado (33,77% de nulos) — segunda melhor cobertura entre todos os indicadores vistos

- **Média:** 71.869 | **Mediana:** 6.436,5 — diferença extrema entre média e mediana, distribuição fortemente assimétrica (concentração muito forte em poucos países, provavelmente China, com volume de marcas registradas ordens de grandeza acima do resto)

- **Dispersão (desvio padrão):** 456.370,8 — mais de 6x a média, a maior dispersão relativa vista até agora entre todos os indicadores

- **Mínimo / Máximo:** 1 / 7.904.365 — intervalo extremo, mas plausível dado o boom de registro de marcas na China nos últimos anos

- **Cobertura por ano:** cresce de forma quase constante — 90 (2004) → 79 (2009, única queda) → 134 (2014) → 148 (2019) → 155 (2024)

- **Duplicatas:** 0 — melt correto

- **Nulo em Country Code:** mesmo caso da Namíbia (0,55%) — não corrigido, conforme decisão de priorização já registrada

**Nota:** este é o indicador com maior concentração observada até agora (desvio padrão 6x a média). Reforça que o Top 10 de marcas provavelmente será dominado por 1-2 países com folga muito grande sobre o resto..

##3.12 Diagnóstico: Pedidos de desenhos industriais

In [194]:
diagnostico_completo(df_desenhos_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,163
Country Code,object,0.61,162
Ano,int64,0.00,5
Valor,float64,36.56,406
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,517.00
mean,9971.02
std,59716.03
min,1.00
25%,55.00
50%,335.00
75%,2267.00
max,907420.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,64
2009,72
2014,117
2019,126
2024,138



== Duplicatas país+ano ==
0


### Observações — Pedidos de desenhos industriais

- **Cobertura:** 517 de 815 combinações possíveis têm dado (36,56% de nulos)

- **Média:** 9.971 | **Mediana:** 335 — diferença grande entre média e mediana, distribuição assimétrica (concentração em poucos países, mesmo padrão dos outros indicadores WIPO)

- **Dispersão (desvio padrão):** 59.716 — cerca de 6x a média, concentração forte, mas um pouco menos extrema que Marcas

- **Mínimo / Máximo:** 1 / 907.420 — intervalo grande, plausível dado o padrão já visto nos outros indicadores WIPO

- **Cobertura por ano:** cresce de forma constante — 64 (2004) → 72 (2009) → 117 (2014) → 126 (2019) → 138 (2024). Esse é o indicador com o crescimento de cobertura mais acentuado (mais que dobrou do início ao fim do período)

- **Duplicatas:** 0 — melt correto
- **Nulo em Country Code:** mesmo caso da Namíbia (0,61%) — não corrigido, mesma decisão de priorização

##3.13 Padronização de nomes de país (comparação + dicionário de correspondência)

Antes de consolidar, precisamos alinhar os nomes de país entre Banco Mundial e WIPO.
Nesta etapa, identificamos de forma sistemática todos os nomes que não batem entre
as duas fontes, para depois construir um dicionário de correspondência.

###Comparar as duas listas de nomes de país

In [195]:
# Conjunto de nomes de país em cada fonte (usamos set() para poder comparar)
paises_bm = set(df_pd_pib["Country Name"].unique())
paises_wipo = set(df_patentes_wipo["Country Name"].unique())

# Nomes que existem no Banco Mundial mas não na WIPO (com esse texto exato)
so_no_bm = sorted(paises_bm - paises_wipo)

# Nomes que existem na WIPO mas não no Banco Mundial (com esse texto exato)
so_na_wipo = sorted(paises_wipo - paises_bm)

print(f"Nomes só no Banco Mundial ({len(so_no_bm)}):")
print(so_no_bm)

print(f"\nNomes só na WIPO ({len(so_na_wipo)}):")
print(so_na_wipo)

Nomes só no Banco Mundial (67):
['Afghanistan', 'American Samoa', 'Aruba', 'Bahamas, The', 'Bermuda', 'Bolivia', 'British Virgin Islands', 'Cayman Islands', 'Channel Islands', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Curacao', 'Czechia', 'Egypt, Arab Rep.', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Gambia, The', 'Gibraltar', 'Greenland', 'Guam', 'Hong Kong SAR, China', 'Iran, Islamic Rep.', 'Isle of Man', 'Kiribati', "Korea, Dem. People's Rep.", 'Korea, Rep.', 'Kosovo', 'Kyrgyz Republic', 'Lao PDR', 'Libya', 'Macao SAR, China', 'Maldives', 'Marshall Islands', 'Micronesia, Fed. Sts.', 'Moldova', 'Myanmar', 'Nauru', 'Netherlands', 'New Caledonia', 'Northern Mariana Islands', 'Palau', 'Puerto Rico (US)', 'Sint Maarten (Dutch part)', 'Slovak Republic', 'Solomon Islands', 'Somalia, Fed. Rep.', 'South Sudan', 'St. Kitts and Nevis', 'St. Lucia', 'St. Martin (French part)', 'St. Vincent and the Grenadines', 'Suriname', 'Tanzania', 'Timor-Leste', 

##Dicionário de correspondência construído

Das 26 divergências de nome na WIPO, todas têm correspondência direta com nomes do
Banco Mundial (apenas grafia/convenção diferente). Os demais 41 nomes exclusivos do
Banco Mundial são territórios/países pequenos sem presença na base WIPO (ex: West
Bank and Gaza, Kosovo, South Sudan, ilhas do Pacífico) — ficam de fora da junção com
WIPO por ausência real de dado, não por erro de nomenclatura.

In [196]:
correspondencia_paises = {
    "Bahamas": "Bahamas, The",
    "Bolivia (Plurinational State of)": "Bolivia",
    "China, Hong Kong SAR": "Hong Kong SAR, China",
    "China, Macao SAR": "Macao SAR, China",
    "Congo": "Congo, Rep.",
    "Czech Republic": "Czechia",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Gambia": "Gambia, The",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Democratic Republic": "Lao PDR",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Slovakia": "Slovak Republic",
    "Türkiye": "Turkiye",
    "United Republic of Tanzania": "Tanzania",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "Yemen": "Yemen, Rep.",
}

print(f"Total de correspondências mapeadas: {len(correspondencia_paises)}")

Total de correspondências mapeadas: 26


In [197]:
# ============================================================
# Padronizar nomes de país nas 3 tabelas WIPO, usando o dicionário
# ============================================================
for d in [df_patentes_wipo, df_marcas_wipo, df_desenhos_wipo]:
    d["Country Name"] = d["Country Name"].replace(correspondencia_paises)

# Confirma que a divergência diminuiu
paises_wipo_ajustado = set(df_patentes_wipo["Country Name"].unique())
print("Nomes ainda só na WIPO (deve ser 0):", sorted(paises_wipo_ajustado - paises_bm))

Nomes ainda só na WIPO (deve ser 0): []


Padronização concluída

Aplicado o dicionário de 26 correspondências nas 3 tabelas WIPO (Patentes, Marcas,
Desenhos Industriais). Confirmado: nenhum nome de país da WIPO ficou fora do padrão
do Banco Mundial após a substituição. As tabelas estão prontas para consolidação.

##3.14 Consolidação das bases

Com os nomes de país padronizados, empilhamos as 7 tabelas (4 do Banco Mundial + 3
da WIPO) em um único dataframe, todas já no mesmo formato long
(Country Name / Country Code / Ano / Valor / Indicador).

In [198]:
# ============================================================
# Consolidação: empilhar as 7 tabelas em um único dataframe
# ============================================================
df_consolidado = pd.concat([
    df_pd_pib,
    df_pesquisadores,
    df_patentes_resd,
    df_exportacoes,
    df_patentes_wipo,
    df_marcas_wipo,
    df_desenhos_wipo
], ignore_index=True)

df_consolidado.shape
df_consolidado["Indicador"].unique()
df_consolidado["Indicador"].value_counts()

,count
Indicador,
Gasto em P&D bm (% do PIB),1085
Pesquisadores em P&D bm (por milhão hab.),1085
Pedidos de patentes bm,1085
Exportações de alta tecnologia bm (%),1085
Pedidos de marcas wipo,915
Pedidos de patentes wipo,880
Pedidos de desenhos industriais wipo,815


In [199]:
# ============================================================
# 3.14.1 Padronização de código de país (Alpha-3)
# O Banco Mundial usa códigos Alpha-3 (ex: DEU, BRA), mas a WIPO usa
# Alpha-2 (ex: DE, BR). Sem padronizar, o mesmo país fica com dois
# valores diferentes de "Country Code" após a consolidação — o que
# infla artificialmente a contagem de países únicos (ex: Sint Maarten
# aparecendo como "SXM" e "SX").
# ============================================================
df_consolidado["Country Code"] = df_consolidado["Country Code"].apply(codigo_para_alpha3)

# Confirma que a padronização resolveu o caso identificado (Sint Maarten)
df_consolidado[df_consolidado["Country Name"].str.contains("Sint Maarten", na=False)][
    ["Country Name", "Country Code"]
].drop_duplicates()

,Country Name,Country Code
184,Sint Maarten (Dutch part),SXM
5367,Sint Maarten (Dutch Part),SXM


###Consolidação confirmada

Dataframe único com os 7 indicadores empilhados. Contagem de linhas por indicador
bate com o esperado: 1085 para cada indicador do Banco Mundial (217 países × 5 anos),
e 915/880/815 para os indicadores WIPO (variação por ausência de registro de
determinado tipo de propriedade intelectual em alguns países).


##3.15 Padronização de nomes de país (tradução para português)

In [200]:
df_consolidado["Country Name PT"] = df_consolidado["Country Code"].apply(traduzir_pais)

df_consolidado[["Country Name", "Country Code", "Country Name PT"]].drop_duplicates(subset="Country Code").head(15)

,Country Name,Country Code,Country Name PT
0,Aruba,ABW,Aruba
1,Afghanistan,AFG,Afeganistão
2,Angola,AGO,Angola
3,Albania,ALB,Albânia
4,Andorra,AND,Andorra
5,United Arab Emirates,ARE,Emirados Árabes Unidos
6,Argentina,ARG,Argentina
7,Armenia,ARM,Armênia
8,American Samoa,ASM,Samoa Americana
9,Antigua and Barbuda,ATG,Antígua e Barbuda


### Tradução de nomes de país concluída

Aplicada a tradução para português usando as bibliotecas `pycountry` (conversão de
código ISO Alpha-3/Alpha-2) e `babel` (tradução oficial de território). A Namíbia
fica sem tradução devido ao problema já documentado de seu código "NA" ser lido
como nulo pelo pandas — sem impacto no escopo do trabalho.

#4. Modelagem (Top 10)



Nesta etapa, gero o ranking Top 10 de cada indicador em todos os 5 anos da frequência escolhida.

O objetivo é obter um panorama de como o ranking mundial de cada indicador evolui ao longo de duas décadas, não uma foto única e fixa do ano base.

A função `top10_por_indicador_ano` é aplicada 35 vezes (7 indicadores × 5 anos),
organizando o resultado numa estrutura única (`top10_evolucao`), que permite consultar
qualquer indicador em qualquer ano sem recalcular nada.


As observações interpretativas detalhadas (Seções 4.2 a 4.8) foram feitas apenas para
o ano-base de cada indicador (2021 para Banco Mundial, 2024 para WIPO)

Os demais 4 anos de cada indicador ficam disponíveis na estrutura `top10_evolucao` para consulta e para a análise de evolução temporal (Seção 4.9), mas não recebem uma célula de texto
dedicada por ano.

##4.1 Geração da estrutura de evolução do Top 10

In [201]:
# ============================================================
# Top 10 por indicador — geração única, cobrindo os 5 anos de cada um
# Resultado organizado em top10_evolucao[indicador][ano]
# ============================================================

indicadores_com_anos = {
    "Gasto em P&D bm (% do PIB)": anos_bm,
    "Pesquisadores em P&D bm (por milhão hab.)": anos_bm,
    "Pedidos de patentes bm": anos_bm,
    "Exportações de alta tecnologia bm (%)": anos_bm,
    "Pedidos de patentes wipo": anos_wipo,
    "Pedidos de marcas wipo": anos_wipo,
    "Pedidos de desenhos industriais wipo": anos_wipo,
}

top10_evolucao = {}

for indicador, anos in indicadores_com_anos.items():
    top10_evolucao[indicador] = {}
    for ano in anos:
        top10_evolucao[indicador][ano] = top10_por_indicador_ano(df_consolidado, indicador, ano)

# Variáveis do ano-base, apontando para dentro da estrutura única
top10_pd_pib = top10_evolucao["Gasto em P&D bm (% do PIB)"][2021]
top10_pesquisadores = top10_evolucao["Pesquisadores em P&D bm (por milhão hab.)"][2021]
top10_patentes_bm = top10_evolucao["Pedidos de patentes bm"][2021]
top10_exportacoes = top10_evolucao["Exportações de alta tecnologia bm (%)"][2021]

top10_patentes_wipo = top10_evolucao["Pedidos de patentes wipo"][2024]
top10_marcas = top10_evolucao["Pedidos de marcas wipo"][2024]
top10_desenhos = top10_evolucao["Pedidos de desenhos industriais wipo"][2024]

# Confirma que gerou tudo: deve mostrar 5 anos por indicador
for indicador in indicadores_com_anos:
    print(f"{indicador}: {len(top10_evolucao[indicador])} anos gerados")

Gasto em P&D bm (% do PIB): 5 anos gerados
Pesquisadores em P&D bm (por milhão hab.): 5 anos gerados
Pedidos de patentes bm: 5 anos gerados
Exportações de alta tecnologia bm (%): 5 anos gerados
Pedidos de patentes wipo: 5 anos gerados
Pedidos de marcas wipo: 5 anos gerados
Pedidos de desenhos industriais wipo: 5 anos gerados


##4.2 Top 10 Gasto em P&D (% do PIB), 2021

In [202]:

top10_pd_pib

,País,Valor
0,Israel,5.76
1,Coreia do Sul,4.60
2,Estados Unidos,3.47
3,Suécia,3.42
4,Bélgica,3.41
5,Japão,3.27
6,Áustria,3.26
7,Suíça,3.25
8,Alemanha,3.08
9,Finlândia,3.01




Israel lidera com folga (5,76%), quase 1,2 ponto percentual à frente do 2º colocado,

Coreia do Sul (4,60%). A partir da 3ª posição, o grupo fica mais concentrado: Estados
Unidos, Suécia, Bélgica, Japão, Áustria, Suíça, Alemanha e Finlândia ocupam a faixa de
3,01% a 3,47%, uma diferença pequena entre si. O ranking é dominado por economias
desenvolvidas da Europa, Ásia e América do Norte — nenhum país em desenvolvimento
aparece.

O valor máximo (5,76%) bate exatamente com o `max` identificado no
diagnóstico estatístico, confirmando consistência entre as etapas.

**Nota:** por ser um indicador percentual (% do PIB), este ranking favorece
economias menores com investimento proporcionalmente alto, não necessariamente
quem mais investe em valor absoluto — a China, por exemplo, tem gasto absoluto
altíssimo, mas seu PIB também é enorme, o que dilui o percentual. Vale ter essa
diferença entre indicadores relativos e absolutos em mente ao comparar os Top 10
desta seção com os de Patentes/Marcas/Desenhos, que são contagens brutas.

##4.3 Top 10 — Pesquisadores em P&D, 2021

In [203]:
top10_pesquisadores

,País,Valor
0,Coreia do Sul,9071.45
1,Suécia,8159.80
2,Singapura,7956.40
3,Finlândia,7870.43
4,Dinamarca,7708.24
5,Noruega,7228.72
6,Islândia,6941.30
7,Áustria,6323.00
8,Países Baixos,6003.87
9,Suíça,6003.41




Coreia do Sul lidera com folga (9.071 pesquisadores por milhão de hab.), cerca de 900
pontos à frente do 2º colocado, Suécia (8.159,8).

Do 2º ao 10º lugar o grupo é mais
coeso — Suécia, Singapura, Finlândia, Dinamarca, Noruega, Islândia, Áustria, Holanda
e Suíça variam entre 6.003 e 8.159, uma faixa de pouco mais de 2.000 pontos.

O ranking é dominado por países pequenos e desenvolvidos (nórdicos, Singapura, Suíça) — o mesmo
padrão de concentração em economias menores já visto no indicador de Gasto em P&D.

**Nota:** por ser um indicador relativo (por milhão de habitantes), favorece países
com população pequena e forte estrutura de pesquisa — Israel, líder em Gasto em P&D,
não aparece aqui, reforçando que os dois indicadores (esforço financeiro vs. densidade
de pesquisadores) não necessariamente elegem os mesmos líderes.

##4.4 Top 10 — Pedidos de patentes (Banco Mundial), 2021

In [204]:
top10_patentes_bm

,País,Valor
0,China,1426644.00
1,Estados Unidos,262244.00
2,Japão,222452.00
3,Coreia do Sul,186245.00
4,Alemanha,39822.00
5,Índia,26267.00
6,Rússia,19569.00
7,França,13386.00
8,Reino Unido,11592.00
9,Itália,10281.00




China lidera com folga extrema: 1.426.644 pedidos, mais de 5x o 2º colocado Estados Unidos (262.244).

Do 2º ao 4º lugar (EUA, Japão, Coreia do Sul) o grupo segue em
ordem de grandeza de centenas de milhares, mas a partir da 5ª posição (Alemanha,
39.822) os valores caem abruptamente para a casa das dezenas de milhares.

 Do 5º ao 10º lugar (Alemanha, Índia, Rússia, França, Reino Unido, Itália) a diferença entre
posições é bem menor, formando um segundo grupo mais coeso.

**Nota:** por ser um indicador absoluto (contagem bruta), este ranking favorece
países com economias grandes —

Diferente do Top 10 de Gasto em P&D (%), aqui China,
EUA e Índia aparecem entre os líderes, o que ilustra o padrão inverso já apontado:
indicadores absolutos elegem países grandes, indicadores relativos elegem países
menores com alta intensidade de investimento.

##4.5 Top 10 — Exportações de alta tecnologia (%), 2021

In [205]:
top10_exportacoes

,País,Valor
0,"Hong Kong, RAE da China",70.55
1,Cuba,70.09
2,Ilhas Cayman,67.77
3,Filipinas,64.23
4,Singapura,54.97
5,Malásia,51.68
6,Vietnã,41.54
7,Papua-Nova Guiné,40.05
8,Coreia do Sul,36.01
9,Islândia,33.49




Hong Kong lidera por pouca margem (70,55%) sobre Cuba (70,09%) e Ilhas Cayman
(67,77%) — os três primeiros ficam bem próximos entre si. Do 4º ao 10º lugar

(Filipinas, Singapura, Malásia, Vietnã, Papua-Nova Guiné, Coreia do Sul, Islândia) os
valores caem de forma mais gradual, de 64,23% a 33,49%.

**Nota sobre a métrica:** este indicador não conta unidades exportadas — mede a
fatia (%) das exportações manufaturadas de cada país que é classificada como alta
tecnologia. Por isso um valor alto reflete concentração da pauta exportadora, não
necessariamente volume absoluto grande.

**Ponto de atenção:** diferente dos Top 10 anteriores, aqui aparecem países não
associados a forte tradição de inovação tecnológica (Cuba, Ilhas Cayman, Papua-Nova
Guiné) ao lado de economias tech conhecidas (Hong Kong, Singapura, Coreia do Sul).
Isso é coerente com a nota acima: economias pequenas com pouca diversificação
industrial podem apresentar percentuais altos mesmo com baixo volume absoluto. Vale
investigar os valores brutos desses casos antes de usar este ranking para conclusões
fortes no relatório.

##4.6 Top 10 — Pedidos de patentes (WIPO), 2024

In [206]:
top10_patentes_wipo

,País,Valor
0,China,1796738.00
1,Estados Unidos,503283.00
2,Japão,420991.00
3,Coreia do Sul,296041.00
4,Alemanha,133790.00
5,Índia,76473.00
6,França,51880.00
7,Reino Unido,46840.00
8,Suíça,41341.00
9,Países Baixos,26445.00




China lidera com folga extrema: 1.796.738 pedidos, mais de 3,5x o 2º colocado,
Estados Unidos (503.283).

 Do 2º ao 4º lugar (EUA, Japão, Coreia do Sul) os valores
seguem na casa das centenas de milhares, com queda gradual.

A partir da 5ª posição
(Alemanha, 133.790) os valores caem para a casa das dezenas de milhares, formando um
segundo grupo (Alemanha, Índia, França, Reino Unido, Suíça, Holanda) mais próximo
entre si.

**Comparação com o mesmo indicador no Banco Mundial (ano 2021):** o ranking e a ordem
dos 8 primeiros países se mantêm praticamente idênticos (China, EUA, Japão, Coreia do
Sul, Alemanha, Índia, França, Reino Unido), com os valores da WIPO em 2024 mais altos
— esperado, já que são anos diferentes e fontes com metodologias próprias de
contagem. A consistência entre as duas fontes reforça a confiabilidade do ranking.

##4.7 Top 10 — Pedidos de marcas (WIPO), 2024

In [207]:
top10_marcas

,País,Valor
0,China,7314677.00
1,Estados Unidos,840047.00
2,Rússia,559679.00
3,Índia,533417.00
4,Brasil,436355.00
5,Alemanha,434215.00
6,Turquia,401096.00
7,França,363218.00
8,Reino Unido,348712.00
9,Japão,340721.00




China lidera de forma avassaladora: 7.314.677 pedidos, quase 9x o 2º colocado,
Estados Unidos (840.047) — a maior distância entre 1º e 2º lugar vista em todos os
Top 10 até agora.

Do 2º ao 10º lugar (EUA, Rússia, Índia, Brasil, Alemanha, Turquia,
França, Reino Unido, Japão) os valores ficam numa faixa mais próxima entre si,
340 mil a 840 mil.



**Brasil aparece pela primeira vez em um Top 10**, na 5ª posição (436.355) — à frente
de economias tradicionalmente associadas a forte atividade de patenteamento, como
Alemanha, França, Reino Unido e Japão.

Isso responde diretamente a uma das perguntas
da Etapa 1 (posição do Brasil frente aos líderes) e é o achado mais relevante do
trabalho até aqui — vale destaque na Fase 2.

##4.7 Top 10 — Pedidos de desenhos industriais (WIPO), 2024

In [208]:
top10_desenhos

,País,Valor
0,China,907420.00
1,Alemanha,70262.00
2,Estados Unidos,67095.00
3,Itália,63709.00
4,Coreia do Sul,60175.00
5,Turquia,44021.00
6,França,42007.00
7,Reino Unido,41370.00
8,Índia,39116.00
9,Japão,33536.00




China lidera com folga extrema: 907.420 pedidos, quase 13x o 2º colocado, Alemanha
(70.262).

Do 2º ao 10º lugar (Alemanha, EUA, Itália, Coreia do Sul, Turquia,
França, Reino Unido, Índia, Japão) os valores ficam numa faixa mais próxima entre
si, de 33 mil a 70 mil.


**Nota:** diferente do Top 10 de Marcas, o Brasil não aparece aqui — mostra que a
posição de destaque do Brasil não se repete de forma uniforme entre os três
indicadores da WIPO, e vale verificar sua posição exata (fora do Top 10) na Fase 2.

##Seção 4.9 Comportamento do Top 10 ao Longo do Tempo

Nesta etapa, comparamos os 5 anos de cada indicador para identificar:
- Quais países permanecem no Top 10 em todos os anos (líderes consistentes)
- Quais países entraram ou saíram do ranking entre o primeiro e o último ano

Essa é a análise transversal que responde à pergunta do professor sobre a evolução
do panorama mundial de cada indicador.

---



In [209]:
for indicador in indicadores_com_anos:
    resumo_evolucao_top10(top10_evolucao, indicador)
    print()

=== Gasto em P&D bm (% do PIB) ===
Países no Top 10 em TODOS os 5 anos (7): ['Alemanha', 'Coreia do Sul', 'Estados Unidos', 'Finlândia', 'Israel', 'Japão', 'Suécia']
Saíram do Top 10 (estavam em 2001, não estão em 2021): ['Dinamarca', 'França', 'Islândia']
Entraram no Top 10 (não estavam em 2001, estão em 2021): ['Bélgica', 'Suíça', 'Áustria']

=== Pesquisadores em P&D bm (por milhão hab.) ===
Países no Top 10 em TODOS os 5 anos (4): ['Dinamarca', 'Noruega', 'Singapura', 'Suécia']
Saíram do Top 10 (estavam em 2001, não estão em 2021): ['Alemanha', 'Canadá', 'Estados Unidos', 'Japão', 'Rússia']
Entraram no Top 10 (não estavam em 2001, estão em 2021): ['Coreia do Sul', 'Finlândia', 'Países Baixos', 'Suíça', 'Áustria']

=== Pedidos de patentes bm ===
Países no Top 10 em TODOS os 5 anos (8): ['Alemanha', 'China', 'Coreia do Sul', 'Estados Unidos', 'França', 'Japão', 'Reino Unido', 'Rússia']
Saíram do Top 10 (estavam em 2001, não estão em 2021): ['Canadá', 'Ucrânia']
Entraram no Top 10 (não

###Comportamento do Top 10 ao longo do tempo

**Gasto em P&D (% do PIB):** 7 de 10 países se mantiveram estáveis em todos os 5
anos (Alemanha, Coreia do Sul, EUA, Finlândia, Israel, Japão, Suécia) — o ranking
mais estável entre todos os indicadores analisados. As trocas (Dinamarca/França/
Islândia saindo, Bélgica/Suíça/Áustria entrando) são movimentos discretos entre
países de perfil semelhante (economias europeias desenvolvidas), não rupturas.

**Pesquisadores em P&D (por milhão hab.):** Apenas 4 países estáveis (Dinamarca,
Noruega, Singapura, Suécia). Chama atenção a saída de potências como Estados
Unidos, Japão, Alemanha e Rússia — sugere que, embora sigam investindo em P&D,
não mantiveram a mesma densidade de pesquisadores por habitante frente a outros
países que cresceram nesse quesito (Coreia do Sul, Suíça, Áustria entrando).

**Pedidos de patentes (Banco Mundial):** 8 de 10 países estáveis — segundo ranking
mais consistente. A principal mudança é a entrada da Índia, coerente com o
crescimento observado nos indicadores WIPO.

**Exportações de alta tecnologia (%):** nenhum país permaneceu no Top 10 nos 5
anos — o ranking mais instável de todo o trabalho. Isso decorre em parte da
limitação de dados já documentada (sem cobertura em 2001 e 2006), mas também
reflete a natureza do indicador: por medir uma fatia percentual da pauta
exportadora, pequenas mudanças na composição de exportação de um país podem
alterar bastante sua posição relativa de um ano para outro.

**Pedidos de patentes (WIPO):** 9 de 10 países estáveis — o ranking mais estável
entre os indicadores WIPO, com apenas a troca Rússia (saiu) por Índia (entrou).

**Pedidos de marcas (WIPO):** Apenas 4 países estáveis (Alemanha, China, EUA,
Rússia). A saída de Coreia do Sul, Suíça, Espanha e Austrália, com a entrada de
França, Japão, Turquia e Índia, sugere um ranking mais disputado nas posições
intermediárias — coerente com o achado de que este indicador tem a maior
concentração nos primeiros lugares (China dominando), deixando mais instabilidade
no restante do ranking.

**Pedidos de desenhos industriais (WIPO):** 6 de 10 países estáveis. Entrada
notável da Índia, mesmo padrão observado nos demais indicadores de patente.

**Padrão geral:** indicadores de patentes (Banco Mundial e WIPO) mostram os
rankings mais estáveis (8-9 de 10 países mantidos), enquanto indicadores
percentuais/relativos (Exportações de alta tecnologia, Pesquisadores por habitante)
mostram maior rotatividade.

 A Índia aparece como entrante em 4 dos 7 indicadores —
o país com ascensão mais consistente ao longo do período analisado.

###  Destaque — Ascensão da Índia

Ao longo da análise de evolução do Top 10 (Seção 4.9), a Índia se destaca como o
único país que **entrou** no ranking em múltiplos indicadores ao longo do período:

- Pedidos de patentes (Banco Mundial): entrou no Top 10 entre 2001 e 2021
- Pedidos de patentes (WIPO): entrou no Top 10 entre 2004 e 2024
- Pedidos de marcas (WIPO): entrou no Top 10 entre 2004 e 2024
- Pedidos de desenhos industriais (WIPO): entrou no Top 10 entre 2004 e 2024

Nenhum outro país aparece como entrante em mais de 1-2 indicadores — a Índia é o
único caso de ascensão consistente e generalizada. Isso é um possível candidato a
recorte adicional para a Fase 2 do trabalho (relatório final).

**Nota:** ainda não decidido se esse recorte entra no escopo final do trabalho —
registrado aqui como observação relevante encontrada durante a análise.

### 4.9.1 Matriz de posição — País × Ano

Para entender com mais detalhe o comportamento do Top 10 ao longo do tempo (além
do resumo de entrada/saída da Seção 4.9), construí uma matriz por indicador,
mostrando a posição exata (1º a 10º) de cada país em cada um dos 5 anos.

Isso permite identificar padrões que a análise anterior não capturava:
- Em qual ano exato um país entrou ou saiu do ranking
- Se um país saiu e depois voltou (intermitência)
- Se a posição de um país dentro do Top 10 subiu, caiu ou se manteve estável
  ao longo do tempo

Células marcadas com "—" indicam que o país não esteve no Top 10 daquele ano.

In [210]:
matrizes_top10 = {}

for indicador in indicadores_com_anos:
    matrizes_top10[indicador] = matriz_posicao_top10(top10_evolucao, indicador)

# Confirma que gerou as 7 matrizes
for indicador, matriz in matrizes_top10.items():
    print(f"{indicador}: {matriz.shape[0]} países distintos ao longo dos 5 anos")

Gasto em P&D bm (% do PIB): 14 países distintos ao longo dos 5 anos
Pesquisadores em P&D bm (por milhão hab.): 18 países distintos ao longo dos 5 anos
Pedidos de patentes bm: 14 países distintos ao longo dos 5 anos
Exportações de alta tecnologia bm (%): 23 países distintos ao longo dos 5 anos
Pedidos de patentes wipo: 11 países distintos ao longo dos 5 anos
Pedidos de marcas wipo: 16 países distintos ao longo dos 5 anos
Pedidos de desenhos industriais wipo: 15 países distintos ao longo dos 5 anos


#### Gasto em P&D (% do PIB)

In [211]:
matrizes_top10["Gasto em P&D bm (% do PIB)"]

,2001,2006,2011,2016,2021
País,,,,,
Alemanha,7,8,7,7,9
Bélgica,—,—,—,10,5
Coreia do Sul,9,6,3,2,2
Dinamarca,8,9,6,6,—
Eslovênia,—,—,10,—,—
Estados Unidos,6,7,8,8,3
Finlândia,3,3,2,9,10
França,10,—,—,—,—
Islândia,5,5,—,—,—


- **Israel**: único país com liderança absoluta e ininterrupta — 1º lugar nos 5 anos, sem exceção.
- **Coreia do Sul**: ascensão mais clara e consistente da matriz — 9º (2001) → 6º → 3º → 2º → 2º (2021), sem nenhum retrocesso.
- **Finlândia**: trajetória inversa — chegou a 2º lugar (2011) e caiu para 9º/10º nos dois últimos recortes, a queda mais acentuada da matriz.
- **Eslovênia e França**: aparições isoladas — Eslovênia entrou só em 2011 (10º) e nunca mais voltou; França esteve em 2001 (10º) e saiu definitivamente logo depois. Nenhum dos dois teve retorno.
- **Bélgica e Suíça**: as duas entradas mais recentes (2016 e 2021, respectivamente) — ainda sem histórico suficiente para avaliar se é tendência de subida ou entrada pontual.
- **Nenhum país voltou ao Top 10** depois de sair definitivamente — não há casos de intermitência (sair e retornar) neste indicador, apenas entradas e saídas permanentes.

####Pesquisadores em P&D (por milhão hab.)

In [212]:
matrizes_top10["Pesquisadores em P&D bm (por milhão hab.)"]

,2001,2006,2011,2016,2021
País,,,,,
Alemanha,10,—,—,—,—
Austrália,—,10,—,—,—
Canadá,6,9,10,—,—
Coreia do Sul,—,—,5,3,1
Dinamarca,7,6,3,1,5
Estados Unidos,8,—,—,—,—
Finlândia,—,2,1,5,4
Irlanda,—,—,—,9,—
Islândia,1,1,2,—,7




- **Suécia e Singapura**: os únicos países com presença nos 5 anos — os verdadeiros
  estáveis deste indicador (a lista de "sempre presentes" da Seção 4.9 havia
  incluído mais 2 países — Dinamarca e Noruega — mas a matriz mostra que ambos
  têm oscilações de posição relevantes, mesmo sem sair do Top 10).
- **Coreia do Sul**: ascensão mais acentuada de todo o trabalho até aqui — fora
  do Top 10 em 2001/2006, entra em 5º (2011) e assume a liderança já em 2021.
- **Islândia**: único caso de intermitência real neste indicador — líder em
  2001/2006, cai e sai do ranking em 2016, mas retorna em 7º em 2021.
- **Alemanha, Estados Unidos e Rússia**: saída precoce e definitiva — os três
  estavam no Top 10 em 2001 e não retornam em nenhum dos 4 recortes seguintes.
- **Maior rotatividade entre os indicadores do Banco Mundial**: 18 países
  distintos ocuparam as 10 posições ao longo do período — mais que o dobro de
  Gasto em P&D e Patentes (14 cada), sugerindo que a "corrida" por densidade de
  pesquisadores é mais disputada e menos concentrada em poucos líderes fixos.

####Pedidos de patentes (Banco Mundial)

In [213]:
matrizes_top10["Pedidos de patentes bm"]

,2001,2006,2011,2016,2021
País,,,,,
Alemanha,4,5,5,5,5
Canadá,10,—,—,—,—
China,5,4,1,1,1
Coreia do Norte,—,9,—,—,—
Coreia do Sul,3,3,4,4,4
Estados Unidos,2,2,3,2,2
França,8,8,8,8,8
Irã,—,10,9,7,—
Itália,—,—,—,—,10




- **China**: a virada mais significativa do indicador — 5º lugar em 2001, assume
  a liderança em 2011 e mantém domínio absoluto até 2021. A ascensão ocorre no
  mesmo período em que o Japão perde a liderança, sugerindo troca direta de
  posições entre os dois países.
- **Japão**: trajetória inversa à da China — líder em 2001, cede a posição
  progressivamente (2º em 2011, 3º em 2016/2021).
- **França e Rússia**: os países com posição mais estável de toda a matriz —
  França oscila só entre 7º-8º, Rússia entre 6º-7º, sem tendência de subida
  ou queda ao longo de 20 anos.
- **Índia**: confirma o padrão de ascensão já identificado — entra em 10º (2011)
  e sobe para 6º em 2021, a maior progressão relativa da matriz depois da China.
- **Menor rotatividade entre indicadores do Banco Mundial**: apenas 14 países
  distintos (empatado com Gasto em P&D), reforçando que os indicadores absolutos
  de patente tendem a ser dominados por um grupo estável de economias grandes.

####Exportações de alta tecnologia (%)

In [214]:
matrizes_top10["Exportações de alta tecnologia bm (%)"]

,2001,2006,2011,2016,2021
País,,,,,
Bermudas,—,—,—,5,—
Cazaquistão,—,—,8,—,—
China,—,—,5,—,—
Chipre,—,—,6,—,—
Coreia do Sul,—,—,7,—,9
Costa Rica,—,—,4,—,—
Cuba,—,—,—,7,2
Filipinas,—,—,—,—,4
"Hong Kong, RAE da China",—,—,—,—,1




- **Sem dado em 2001 e 2006**: confirma a limitação já documentada na Seção 3
  (diagnóstico). Na prática, esta matriz reflete apenas 3 recortes reais
  (2011, 2016, 2021), não 5.
- **Nenhum país esteve presente nos 3 anos disponíveis exceto Singapura**
  (1º em 2011 → 3º em 2016 → 5º em 2021) — trajetória de queda gradual, mas
  ainda assim o único caso de continuidade real no indicador.
- **São Tomé e Príncipe**: caso mais extremo do trabalho — assume a liderança em
  2016 e desaparece completamente do ranking nos demais anos, sem qualquer
  outro registro.
- **Hong Kong**: entra direto na liderança em 2021, sem aparição nos anos
  anteriores disponíveis.
- **23 países distintos disputando 10 posições em apenas 3 recortes com dado**
  — a maior dispersão de todo o trabalho, reforçando a observação já feita
  (Seção 4.9) de que este é o indicador com o ranking mundial mais volátil.
  Combinado com a natureza do próprio indicador (fatia percentual da pauta
  exportadora, sensível à composição do que cada país exporta), este é o
  indicador menos indicado para conclusões sobre lideranças "consolidadas" em
  inovação tecnológica.

####Pedidos de patentes (WIPO)

In [215]:
matrizes_top10["Pedidos de patentes wipo"]

,2004,2009,2014,2019,2024
País,,,,,
Alemanha,4,5,5,5,5
China,5,3,1,1,1
Coreia do Sul,3,4,4,4,4
Estados Unidos,2,2,2,2,2
França,6,6,6,6,7
Japão,1,1,3,3,3
Países Baixos,8,10,9,10,10
Reino Unido,7,7,7,7,8
Rússia,9,9,10,—,—




- **A matriz mais estável do trabalho**: apenas 11 países ocuparam as 10
  posições em 20 anos — a menor rotatividade entre os 7 indicadores.
- **Estados Unidos**: único país com posição 100% fixa (2º lugar) nos 5 anos.
- **China e Japão**: confirmam, com uma segunda fonte de dados, a mesma troca de
  liderança já identificada em Patentes (Banco Mundial) — China assume o 1º
  lugar em 2014 exatamente quando o Japão perde a liderança que mantinha desde
  2004.
- **Rússia e Índia**: movimento espelhado — Rússia sai definitivamente após
  2014, e a Índia entra logo em seguida (2019), como se ocupasse o espaço
  aberto. Essa é a terceira confirmação da ascensão da Índia identificada neste
  trabalho (após Patentes BM e a análise de estabilidade da Seção 4.9).
- **Padrão consistente entre fontes**: a convergência entre esta matriz e a de
  Patentes (Banco Mundial) — mesmos países, mesma direção de movimento, mesmo
  ponto de virada — reforça a confiabilidade dos dois indicadores como retrato
  real da dinâmica de patenteamento mundial.

####Pedidos de marcas (WIPO)

In [216]:
matrizes_top10["Pedidos de marcas wipo"]

,2004,2009,2014,2019,2024
País,,,,,
Alemanha,3,3,3,4,6
Austrália,10,—,—,—,—
Brasil,9,10,—,—,5
China,1,1,1,1,1
Coreia do Sul,4,6,10,9,—
Espanha,8,—,—,—,—
Estados Unidos,2,2,2,2,2
França,—,—,4,6,8
Irã,—,—,—,5,—




- **China**: liderança absoluta e ininterrupta — 1º lugar nos 5 anos, sem
  nenhuma variação. Junto com Israel (Gasto em P&D), é um dos únicos dois casos
  de domínio 100% constante em todo o trabalho.
- **Estados Unidos**: 2º lugar fixo nos 5 anos, sem oscilação — mesmo padrão de
  posição travada já visto em Patentes (WIPO).
- **Brasil**: o caso de intermitência mais notável do trabalho — presente em
  2004/2009 (9º/10º), ausente em 2014/2019, e retorna em 2024 em 5º lugar —
  melhor posição do que jamais teve nos anos anteriores. Diferente de outros
  casos de retorno identificados neste trabalho (que voltam em posição pior),
  o Brasil retorna em ascensão.
- **Coreia do Sul**: trajetória oposta à do Brasil no mesmo período — 4º (2004)
  cai progressivamente até sair do ranking em 2024.
- **Rússia**: presença constante nos 5 anos (junto com China e EUA), mas com a
  posição mais instável dos três (6º-10º-3º), sem tendência de subida ou queda.

####Pedidos de desenhos industriais (WIPO)

In [217]:
matrizes_top10["Pedidos de desenhos industriais wipo"]

,2004,2009,2014,2019,2024
País,,,,,
Alemanha,2,2,2,2,2
Austrália,10,—,—,—,—
China,1,1,1,1,1
Coreia do Sul,5,3,3,3,5
Espanha,—,9,9,—,—
Estados Unidos,6,6,5,4,3
França,4,—,—,6,7
Itália,—,4,4,5,4
Japão,3,5,7,8,10




- **China**: liderança absoluta nos 5 anos — o terceiro indicador (junto com
  Gasto em P&D/Israel e Marcas/China) com domínio 100% constante no 1º lugar.
- **Alemanha**: 2º lugar cravado nos 5 anos, sem nenhuma variação — o terceiro
  caso de "segunda posição travada" identificado no trabalho (após EUA em
  Patentes e Marcas WIPO).
- **Japão**: a queda mais longa e ininterrupta do trabalho — 3º lugar em 2004
  até 10º em 2024, perdendo posição em todos os recortes sem nenhuma
  recuperação no meio do caminho.
- **Turquia**: presença constante nos 5 anos com a menor variação de posição
  entre os países "do meio do pelotão" (oscila só entre 6º-7º).
- **Índia**: entra pela primeira vez apenas em 2024 (9º) — a mais tardia e
  discreta das 4 aparições da Índia identificadas neste trabalho, mas ainda
  assim reforçando o padrão de ascensão já documentado.

####Síntese — Padrões que atravessam múltiplos indicadores



**Domínio 100% constante (1º lugar nos 5 anos, sem exceção):**
Israel (Gasto em P&D), China (Marcas e Desenhos Industriais) — os únicos 3 casos
de liderança absoluta e ininterrupta em todo o trabalho.

**"Segunda posição travada" (2º lugar fixo nos 5 anos):**
Estados Unidos (Patentes WIPO, Marcas), Alemanha (Desenhos Industriais) — um
padrão curioso de países que, mesmo sem ameaçar o líder, também não perdem a
vice-liderança para ninguém.

**A ascensão da Índia é o padrão mais consistente entre indicadores:**
confirmada em 4 dos 7 (Patentes BM, Patentes WIPO, Marcas, Desenhos Industriais)
— nenhum outro país aparece como entrante em tantos indicadores diferentes.

**China e Japão: a troca de liderança mais documentada do trabalho:**
identificada de forma independente tanto em Patentes (Banco Mundial, virada em
2011) quanto em Patentes (WIPO, virada em 2014) — a mesma dinâmica confirmada
por duas fontes de dados diferentes.

**Brasil: único caso de retorno em posição melhor:**
saiu do Top 10 de Marcas (2014-2019) e voltou em 2024 numa posição superior à
que tinha antes de sair — padrão distinto de todos os outros casos de retorno
identificados (que voltam em posição igual ou pior).

**Instabilidade fortemente ligada à natureza do indicador:**
indicadores absolutos (contagem bruta: Patentes, Marcas, Desenhos) mostram os
rankings mais estáveis; indicadores relativos/percentuais (Exportações de alta
tecnologia, Pesquisadores por habitante) mostram a maior rotatividade —
particularmente Exportações de alta tecnologia, o único indicador sem nenhum
país presente em todos os anos com dado disponível.

## 4.10 Top 10 com produção mundial (para tabelas finais)



Para os indicadores absolutos (contagem bruta: Patentes, Marcas, Desenhos
Industriais), geramos uma versão do Top 10 com duas informações adicionais: a
fatia (%) de cada país na produção mundial total, e uma linha "Total mundial"
com a soma de todos os países que reportaram dado naquele ano — não só os 10
exibidos. Indicadores percentuais/per-capita (Gasto em P&D, Exportações,
Pesquisadores) não recebem essas colunas, já que "somar percentuais" não tem
significado real.

In [218]:
abreviacoes = {
    "Gasto em P&D bm (% do PIB)": "Gasto_PeD",
    "Pesquisadores em P&D bm (por milhão hab.)": "Pesquisadores",
    "Pedidos de patentes bm": "Patentes_BM",
    "Exportações de alta tecnologia bm (%)": "Exportacoes",
    "Pedidos de patentes wipo": "Patentes_WIPO",
    "Pedidos de marcas wipo": "Marcas_WIPO",
    "Pedidos de desenhos industriais wipo": "Desenhos_WIPO",
}

In [219]:
tabelas_todos_anos = {}

for indicador, anos in indicadores_com_anos.items():
    for ano in anos:
        chave = f"{indicador} — {ano}"
        bruta = top10_com_total_mundial(df_consolidado, indicador, ano)
        tabelas_todos_anos[chave] = formatar_para_exibicao(bruta, indicador)

print(f"Total de tabelas geradas: {len(tabelas_todos_anos)}")

with pd.ExcelWriter(CAMINHO + "top10_todos_os_anos.xlsx", engine="openpyxl") as writer:
    for indicador, anos in indicadores_com_anos.items():
        for ano in anos:
            chave = f"{indicador} — {ano}"
            nome_aba = f"{abreviacoes[indicador]}_{ano}"[:31]
            tabela = tabelas_todos_anos[chave]

            titulo = pd.DataFrame([[f"{indicador} — {ano}"]])
            titulo.to_excel(writer, sheet_name=nome_aba, index=False, header=False, startrow=0)
            tabela.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2)

print("Excel exportado para:", CAMINHO + "top10_todos_os_anos.xlsx")

Total de tabelas geradas: 35
Excel exportado para: /content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/top10_todos_os_anos.xlsx


# 5. Limpeza final da base consolidada

*Nota: por conveniência de fluxo de trabalho, esta limpeza final é feita após a
Modelagem (Seção 4), não imediatamente após a Seção 3 — onde conceitualmente
pertenceria.*

Antes de encerrar este notebook, removemos a coluna `Country Name` (nome do país
em inglês) — ela serviu apenas como chave temporária para a padronização de nomes
entre Banco Mundial e WIPO (Seção 3.13), e não é mais necessária agora que a
tradução para português (`Country Name PT`) já está aplicada a todos os registros.

As colunas restantes foram renomeadas para português e reordenadas, deixando a
base pronta para exportação: **País / Código País / Ano / Valor / Indicador**.

Esta é a última modificação na base neste notebook. A partir daqui, seguimos para
a exportação dos dados e estruturas geradas, que serão consumidos no Notebook 2
(Correlação, Avaliação e Implantação).

In [220]:
# ============================================================
# Limpeza final do df_consolidado antes da exportação
# Remove a coluna de nome em inglês (já cumpriu seu papel na padronização)
# e reordena as colunas: País / Código País / Ano / Valor / Indicador
# ============================================================

df_consolidado = df_consolidado.drop(columns=["Country Name"])
df_consolidado = df_consolidado.rename(columns={
    "Country Name PT": "País",
    "Country Code": "Código País"
})

df_consolidado = df_consolidado[["País", "Código País", "Ano", "Valor", "Indicador"]]

df_consolidado.head()
df_consolidado.shape

(6950, 5)

In [221]:
df_consolidado.head()

,País,Código País,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Gasto em P&D bm (% do PIB)
1,Afeganistão,AFG,2021,NaN,Gasto em P&D bm (% do PIB)
2,Angola,AGO,2021,NaN,Gasto em P&D bm (% do PIB)
3,Albânia,ALB,2021,0.19,Gasto em P&D bm (% do PIB)
4,Andorra,AND,2021,NaN,Gasto em P&D bm (% do PIB)


In [222]:
df_consolidado.shape

(6950, 5)

## 5.1 Enriquecimento geográfico e por nível de renda

Para investigar a hipótese de uma "virada tecnológica" em direção à Ásia (levantada
durante o trabalho), adicionei duas colunas à base consolidada: o continente
geográfico de cada país e sua classificação por nível de renda (Banco Mundial),
permitindo agregações e rankings por região e por grupo de desenvolvimento nas
próximas etapas.

In [223]:
df_consolidado["Continente"] = df_consolidado["Código País"].apply(continente_do_pais)

df_consolidado[["País", "Código País", "Continente"]].drop_duplicates(subset="Código País")["Continente"].value_counts(dropna=False)

,count
Continente,
África,54
Ásia,49
Europa,46
América do Norte,33
Oceania,19
América do Sul,12
None,5


In [224]:
sem_continente = df_consolidado[df_consolidado["Continente"].isna()][["País", "Código País"]].drop_duplicates()
sem_continente

,País,Código País
34,CHI,CHI
184,Sint Maarten,SXM
193,Timor-Leste,TLS
212,XKX,XKX
4450,None,None


### Países sem continente identificado

8 registros de país (de ~404) não tiveram continente identificado automaticamente
pela biblioteca `pycountry_convert`:

- **Namíbia** — mesmo problema já documentado (código "NA" lido como nulo pelo pandas)
- **Timor-Leste** — corrigido manualmente (Ásia)
- **Channel Islands, Sint Maarten (2 códigos), XKX/Kosovo** — territórios/códigos
  não-padrão que a biblioteca não reconhece
- **AN (Antilhas Holandesas) e YU (Iugoslávia)** — códigos históricos de países/
  territórios que deixaram de existir (extintos em 2010 e 2003, respectivamente),
  presentes nos primeiros anos-base da série

Nenhum desses casos representa um país com relevância para os rankings/Top 10 do
trabalho, então não foram corrigidos individualmente — ficam sem valor de
Continente nas agregações regionais.

In [225]:
df_consolidado.loc[df_consolidado["Código País"] == "TLS", "Continente"] = "Ásia"

# Confirma que Timor-Leste agora tem continente atribuído
df_consolidado[df_consolidado["Código País"] == "TLS"][["País", "Código País", "Continente"]].drop_duplicates()

,País,Código País,Continente
193,Timor-Leste,TLS,Ásia


### Enriquecimento concluído

Base consolidada agora inclui `Continente` (geográfico) e `Nível de Renda`
(classificação do Banco Mundial, traduzida para desenvolvido/emergente/
subdesenvolvido), com cobertura de aproximadamente 99% dos registros em ambas
as colunas — as poucas exceções são territórios pequenos ou códigos históricos
de países extintos (Iugoslávia, Antilhas Holandesas), sem impacto relevante
para os rankings do trabalho.

#5.2 Indicadores por nível de renda

In [226]:


# Cria uma coluna auxiliar só para o merge, sem alterar Código País original
df_consolidado["_codigo_alpha3"] = df_consolidado["Código País"].apply(codigo_para_alpha3)

nivel_renda = carregar_nivel_renda(
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/metadata_country_pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv"
)
nivel_renda = nivel_renda.rename(columns={"Código País": "_codigo_alpha3"})

df_consolidado = df_consolidado.merge(nivel_renda, on="_codigo_alpha3", how="left")
df_consolidado = df_consolidado.drop(columns=["_codigo_alpha3"])

df_consolidado["Nível de Renda"].value_counts(dropna=False)

,count
Nível de Renda,
Renda alta (desenvolvido),2665
Renda média-alta (emergente),1920
Renda média-baixa (emergente),1535
Renda baixa (subdesenvolvido),805
NaN,25


Complementando o recorte geográfico (Seção anterior), separei cada indicador por
nível de renda (desenvolvido/emergente-alta/emergente-baixa/subdesenvolvido), em
todos os 5 anos, para verificar se a concentração observada por continente também
se reflete numa concentração por grupo de desenvolvimento econômico.

In [227]:
ranking_renda_evolucao = {}

for indicador, anos in indicadores_com_anos.items():
    ranking_renda_evolucao[indicador] = {}
    for ano in anos:
        ranking_renda_evolucao[indicador][ano] = ranking_nivel_renda(df_consolidado, indicador, ano)

        print(f"\n{indicador} — {ano}")
        display(ranking_renda_evolucao[indicador][ano])


Gasto em P&D bm (% do PIB) — 2021


,Ranking,Nível de Renda,Valor
0,1,Renda alta (desenvolvido),1.74
1,2,Renda média-alta (emergente),0.47
2,3,Renda média-baixa (emergente),0.24
3,4,Renda baixa (subdesenvolvido),0.20



Gasto em P&D bm (% do PIB) — 2016


,Ranking,Nível de Renda,Valor
0,1,Renda alta (desenvolvido),1.48
1,2,Renda média-alta (emergente),0.47
2,3,Renda baixa (subdesenvolvido),0.30
3,4,Renda média-baixa (emergente),0.27



Gasto em P&D bm (% do PIB) — 2011


...

,Ranking,Nível de Renda,Valor,% da produção mundial
0,1,Renda média-alta (emergente),635110.00,58.31
1,2,Renda alta (desenvolvido),438212.00,40.23
2,3,Renda média-baixa (emergente),15063.00,1.38
3,4,Renda baixa (subdesenvolvido),806.00,0.07



Pedidos de desenhos industriais wipo — 2009


,Ranking,Nível de Renda,Valor,% da produção mundial
0,1,Renda média-alta (emergente),389658.00,52.84
1,2,Renda alta (desenvolvido),339802.00,46.08
2,3,Renda média-baixa (emergente),7475.00,1.01
3,4,Renda baixa (subdesenvolvido),482.00,0.07



Pedidos de desenhos industriais wipo — 2004


,Ranking,Nível de Renda,Valor,% da produção mundial
0,1,Renda alta (desenvolvido),277864.00,65.91
1,2,Renda média-alta (emergente),137217.00,32.55
2,3,Renda média-baixa (emergente),6053.00,1.44
3,4,Renda baixa (subdesenvolvido),461.00,0.11


1. Gasto em P&D (% PIB) — esforço
Hierarquia estável em todos os 5 anos: Renda alta sempre em 1º, com folga enorme (1,34–1,74% vs. os demais, que nunca passam de 0,47%). Entre os grupos de baixa renda há uma inversão interessante: em 2001 e 2011 a renda média-baixa supera a renda baixa, mas em 2006 e 2016 a renda baixa assume o 3º lugar — ranking instável no fundo da tabela, mas a distância pro topo (renda alta) só aumenta.

2. Pesquisadores em P&D (por milhão hab.) — esforço
Mesmo padrão do gasto: renda alta 1º isolado e crescendo (2.359 → 4.082, quase dobrou), renda média-alta 2º estável. Aqui não há inversão entre os dois últimos grupos — renda média-baixa sempre à frente de renda baixa, que segue com valores irrisórios (29 → 28,67, praticamente estagnado em 20 anos).

3. Patentes BM — resultado
Aqui está a virada mais marcante do conjunto: renda alta domina isolada em 2001 (93,61% da produção mundial) e ainda lidera em 2006 e 2011, mas perde a liderança para renda média-alta a partir de 2016 (58,44% vs 40,80%) e a diferença se amplia em 2021 (63,82% vs 34,93%). É a métrica de resultado onde a renda média-alta mais cresce.

4. Exportações de alta tecnologia (%) — resultado
Série mais curta (só 2011, 2016, 2021) e com um detalhe que quebra o padrão dos outros indicadores: em 2011, renda baixa aparece em 2º lugar (8,09%), à frente até da renda média-alta. Depois volta ao padrão esperado (renda alta 1º, renda média-alta 2º) em 2016 e 2021, com renda média-baixa sempre por último.

5. Patentes WIPO — resultado
Confirma a mesma virada das patentes BM, só que ainda mais nítida: renda alta domina com 93% em 2004, mas cai para 2º lugar já em 2024 (47,54% vs 50,09% de renda média-alta) — a inversão de liderança acontece exatamente no mesmo período (meados/fim dos 2010) que nas patentes BM.

##5.2.1 Ranking dos Continentes

In [228]:
ranking_continentes_evolucao = {}

for indicador, anos in indicadores_com_anos.items():
    ranking_continentes_evolucao[indicador] = {}
    for ano in anos:
        ranking_continentes_evolucao[indicador][ano] = ranking_continentes(df_consolidado, indicador, ano)

        print(f"\n{indicador} — {ano}")
        display(ranking_continentes_evolucao[indicador][ano])


Gasto em P&D bm (% do PIB) — 2021


,Ranking,Continente,Valor
0,1,Oceania,1.67
1,2,Europa,1.60
2,3,Ásia,1.04
3,4,América do Norte,0.74
4,5,África,0.46
5,6,América do Sul,0.43



Gasto em P&D bm (% do PIB) — 2016


,Ranking,Continente,Valor
0,1,Europa,1.40
1,2,Ásia,0.90
2,3,América do Norte,0.68
3,4,América do Sul,0.43
4,5,África,0.36
5,6,Oceania,0.03



Gasto em P&D bm (% do PIB) — 2011


...

,Ranking,Continente,Valor,% da produção mundial
0,1,Ásia,746884.00,68.57
1,2,Europa,266734.00,24.49
2,3,América do Norte,55463.00,5.09
3,4,África,8833.00,0.81
4,5,América do Sul,6084.00,0.56
5,6,Oceania,5193.00,0.48



Pedidos de desenhos industriais wipo — 2009


,Ranking,Continente,Valor,% da produção mundial
0,1,Ásia,485148.00,65.79
1,2,Europa,203130.00,27.55
2,3,América do Norte,35898.00,4.87
3,4,América do Sul,5895.00,0.80
4,5,África,3680.00,0.50
5,6,Oceania,3666.00,0.50



Pedidos de desenhos industriais wipo — 2004


,Ranking,Continente,Valor,% da produção mundial
0,1,Ásia,233154.00,55.30
1,2,Europa,146285.00,34.70
2,3,América do Norte,29511.00,7.00
3,4,Oceania,5112.00,1.21
4,5,América do Sul,4554.00,1.08
5,6,África,2979.00,0.71


### Ranking de continentes — a virada tecnológica é real, mas assimétrica

Os dados confirmam a hipótese de uma virada tecnológica em direção à Ásia — mas de
forma mais matizada do que "a Ásia lidera em tudo": o continente domina crescentemente
o **resultado** da inovação, sem liderar o **esforço** medido nos indicadores
relativos.

**Domínio asiático nos indicadores absolutos (resultado), com participação mundial
crescente em todos os 5 anos:**
- Pedidos de patentes (Banco Mundial): 59,89% (2001) → **82,63%** (2021)
- Pedidos de patentes (WIPO): 51,04% (2004) → **72,42%** (2024)
- Pedidos de desenhos industriais: 55,30% (2004) → **72,44%** (2024)
- Pedidos de marcas: 29,12% (2004) → **66,39%** (2024), ultrapassando a Europa
  (que liderava em 2004 e 2009)

Em nenhum dos quatro indicadores absolutos a Ásia deixou de crescer sua fatia da
produção mundial entre o primeiro e o último recorte — o oposto do padrão observado
nos indicadores de esforço.

**Ásia competitiva, mas não líder, nos indicadores de esforço (relativos):**
- Gasto em P&D (% do PIB): Ásia sempre em 3º lugar nos 5 anos, atrás de Europa e
  Oceania — este último grupo pequeno de países (puxado por Austrália/Nova Zelândia)
  distorce a média por ter poucos membros com valores altos
- Pesquisadores por milhão de habitantes: mesmo padrão — Ásia em 3º-4º lugar,
  nunca na liderança
- Exportações de alta tecnologia (%): Ásia lidera apenas em 2021 (13,85%, por
  margem estreita sobre a Europa); nos demais anos com dado, fica em 2º-3º

**Leitura para o trabalho:** a "virada tecnológica" observada não é de investimento
per capita ou proporcional ao PIB — é de **volume de produção**. A Ásia, puxada
sobretudo pela China (já identificada como líder absoluto em todos os 4 indicadores
de propriedade intelectual nas matrizes do Notebook 1), converte escala populacional
e econômica em resultado bruto, sem necessariamente ostentar a maior intensidade
relativa de investimento em P&D — reforçando o padrão já discutido nas Seções 4:
indicadores absolutos favorecem países/regiões grandes, indicadores relativos
favorecem economias menores com alta intensidade de investimento.

In [229]:
# Exportação: ranking de continentes
with pd.ExcelWriter(CAMINHO + "ranking_continentes.xlsx", engine="openpyxl") as writer:
    for indicador, anos in indicadores_com_anos.items():
        for ano in anos:
            tabela = ranking_continentes_evolucao[indicador][ano]
            nome_aba = f"{abreviacoes[indicador]}_{ano}"[:31]

            titulo = pd.DataFrame([[f"{indicador} — {ano}"]])
            titulo.to_excel(writer, sheet_name=nome_aba, index=False, header=False, startrow=0)
            tabela.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2)

print("Excel exportado para:", CAMINHO + "ranking_continentes.xlsx")

# Exportação: ranking por nível de renda
with pd.ExcelWriter(CAMINHO + "ranking_nivel_renda.xlsx", engine="openpyxl") as writer:
    for indicador, anos in indicadores_com_anos.items():
        for ano in anos:
            tabela = ranking_renda_evolucao[indicador][ano]
            nome_aba = f"{abreviacoes[indicador]}_{ano}"[:31]

            titulo = pd.DataFrame([[f"{indicador} — {ano}"]])
            titulo.to_excel(writer, sheet_name=nome_aba, index=False, header=False, startrow=0)
            tabela.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2)

print("Excel exportado para:", CAMINHO + "ranking_nivel_renda.xlsx")

Excel exportado para: /content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/ranking_continentes.xlsx
Excel exportado para: /content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/ranking_nivel_renda.xlsx


#6. Exportação para o Notebook 2

Este notebook cobre as Etapas 1 a 4 do CRISP-DM (Entendimento do Negócio,
Entendimento dos Dados, Preparação e Modelagem). A partir daqui, o trabalho
continua no notebook `02_correlacao_avaliacao_implantacao.ipynb`, que cobre
as Etapas 5 e 6 (Avaliação e Implantação) — incluindo a análise de correlação
entre esforço (P&D) e resultado (patentes/exportações), conforme a pergunta (b)
do guia do trabalho, e a exportação das tabelas finais para o formato docx.

Arquivos exportados nesta etapa:
- `df_consolidado.csv` — base consolidada, já traduzida e limpa
- `top10_evolucao.pkl` — Top 10 de cada indicador, nos 5 anos de cada frequência
- `matrizes_top10.pkl` — matriz de posição (país × ano) de cada indicador

In [230]:
# ============================================================
# 6. Exportação para o próximo notebook (Correlação, Avaliação e Implantação)
# ============================================================
import pickle

# Base consolidada -> CSV (formato tabular simples)
df_consolidado.to_csv(CAMINHO + "df_consolidado.csv", index=False)

# Estruturas em dicionário -> pickle (preserva a estrutura exata)
with open(CAMINHO + "top10_evolucao.pkl", "wb") as f:
    pickle.dump(top10_evolucao, f)

with open(CAMINHO + "matrizes_top10.pkl", "wb") as f:
    pickle.dump(matrizes_top10, f)

print("Arquivos exportados com sucesso para:", CAMINHO)

Arquivos exportados com sucesso para: /content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/
